In [5]:
import importlib
import src.forward_model
importlib.reload(src.forward_model)

<module 'src.forward_model' from 'c:\\Users\\bayer\\PycharmProjects\\TMStim\\src\\forward_model.py'>

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path

from src.forward_model import vector_potential_loop

save_dir = Path("saves")

# -----------------------------
# Figure-8 coil approximation
# -----------------------------
R = 0.035          # loop radius: 35 mm for 70 mm diameter winding
d = 0.035         # half-distance between loop centers
z0 = 0.02         # cortical plane 20 mm below coil
Nseg = 300        # current loop discretization

# Grid in meters
x = np.linspace(-0.09, 0.09, 160)
y = np.linspace(-0.09, 0.09, 160)
X, Y = np.meshgrid(x, y)
Z = -z0 * np.ones_like(X)

# Two loops with opposite current directions
Ax1, Ay1, Az1 = vector_potential_loop(X, Y, Z, center_x=-d, current_sign=1)
Ax2, Ay2, Az2 = vector_potential_loop(X, Y, Z, center_x= d, current_sign=-1)

Ax = Ax1 + Ax2
Ay = Ay1 + Ay2
Az = Az1 + Az2

# For sinusoidal/time-varying current, E = -dA/dt.
# Ignoring global scale, E-field profile magnitude is proportional to |A|.
E_mag = np.sqrt(Ax**2 + Ay**2 + Az**2)

# Normalize for plotting
E_mag = E_mag / np.max(E_mag)

# Convert axes to mm
Xmm = X * 1000
Ymm = Y * 1000

fig = go.Figure()

fig.add_trace(
    go.Surface(
        x=Xmm,
        y=Ymm,
        z=E_mag,
        colorscale="Viridis",
        colorbar=dict(title="Normalized |E|"),
        contours={
            "z": {
                "show": True,
                "usecolormap": True,
                "highlightcolor": "white",
                "project_z": True
            }
        }
    )
)

fig.update_layout(
    title="Approximate Induced E-field Profile of a Figure-8 70 mm Coil",
    scene=dict(
        xaxis_title="x [mm]",
        yaxis_title="y [mm]",
        zaxis_title="Normalized |E|",
        camera=dict(
            eye=dict(x=1.7, y=-1.7, z=1.1)
        )
    ),
    width=900,
    height=700
)

fig.write_html(
    save_dir / "figure8_efield.html",
    include_plotlyjs="cdn"
)

In [10]:
import webbrowser
webbrowser.open(save_dir / "figure8_efield.html")

True

In [ ]:
import numpy as np
import plotly.graph_objects as go

# -----------------------------
# Geometry
# -----------------------------
R = 0.035       # loop radius [m]
d = 0.035       # half center separation [m]
z_plane = -0.02 # field/evaluation plane [m]

# Coil curve
theta = np.linspace(0, 2*np.pi, 500)

x_left = -d + R*np.cos(theta)
y_left = R*np.sin(theta)
z_left = np.zeros_like(theta)

x_right = d + R*np.cos(theta)
y_right = R*np.sin(theta)
z_right = np.zeros_like(theta)

# -----------------------------
# Reuse approximate field model
# -----------------------------
Nseg = 250
theta_seg = np.linspace(0, 2*np.pi, Nseg, endpoint=False)
dtheta = theta_seg[1] - theta_seg[0]

x = np.linspace(-0.09, 0.09, 120)
y = np.linspace(-0.09, 0.09, 120)
X, Y = np.meshgrid(x, y)
Z = z_plane * np.ones_like(X)

Ax1, Ay1, Az1 = vector_potential_loop(X, Y, Z, center_x=-d, current_sign=1)
Ax2, Ay2, Az2 = vector_potential_loop(X, Y, Z, center_x= d, current_sign=-1)

Ax = Ax1 + Ax2
Ay = Ay1 + Ay2
Az = Az1 + Az2

E = np.sqrt(Ax**2 + Ay**2 + Az**2)
E = E / E.max()

# Convert to mm
mm = 1000
Xmm = X * mm
Ymm = Y * mm
Zmm = Z * mm

# -----------------------------
# Plot
# -----------------------------
fig = go.Figure()

# Left coil
fig.add_trace(
    go.Scatter3d(
        x=x_left*mm,
        y=y_left*mm,
        z=z_left*mm,
        mode="lines",
        line=dict(width=8),
        name="Left winding"
    )
)

# Right coil
fig.add_trace(
    go.Scatter3d(
        x=x_right*mm,
        y=y_right*mm,
        z=z_right*mm,
        mode="lines",
        line=dict(width=8),
        name="Right winding"
    )
)

# Centers
fig.add_trace(
    go.Scatter3d(
        x=[-d*mm, d*mm, 0],
        y=[0, 0, 0],
        z=[0, 0, z_plane*mm],
        mode="markers+text",
        marker=dict(size=5),
        text=["Left center", "Right center", "Target plane center"],
        textposition="top center",
        name="Reference points"
    )
)

# Evaluation plane with E-field contours
fig.add_trace(
    go.Surface(
        x=Xmm,
        y=Ymm,
        z=Zmm,
        surfacecolor=E,
        colorscale="Viridis",
        opacity=0.75,
        colorbar=dict(title="Normalized |E|"),
        contours=dict(
            z=dict(show=False),
            x=dict(show=False),
            y=dict(show=False)
        ),
        name="Evaluation plane"
    )
)

# Add contour lines on the same plane
fig.add_trace(
    go.Surface(
        x=Xmm,
        y=Ymm,
        z=Zmm,
        surfacecolor=E,
        colorscale="Viridis",
        opacity=0.75,
        colorbar=dict(title="Normalized |E|"),
        contours=dict(
            z=dict(
                show=True,
                usecolormap=True,
                project_z=True,
                highlightcolor="white"
            )
        ),
        name="Evaluation plane"
    )
)

fig.update_layout(
    title="Figure-8 Coil Geometry and E-field Evaluation Plane",
    scene=dict(
        xaxis_title="x [mm]",
        yaxis_title="y [mm]",
        zaxis_title="z [mm]",
        aspectmode="data",
        camera=dict(
            eye=dict(x=1.7, y=-1.8, z=1.1)
        )
    ),
    width=950,
    height=750
)

fig.write_html(
    save_dir / "figure8_coil.html",
    include_plotlyjs="cdn"
)

In [16]:
import webbrowser
webbrowser.open(save_dir / "figure8_coil.html")

True